In [15]:
from Environment import *
from DDPG import *
from NN_Module import *
from config_56bus import Config
import os

import torch
import matplotlib.pyplot as plt
import scienceplots
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import plotly.colors as pc
from scipy import interpolate

from loguru import logger
import os
import numpy as np

# plt.style.use('ieee')
# Update global plot parameters

# 方案1 - 柔和专业的配色
# colors = ['#4477AA', '#66CCEE', '#228833', '#CCBB44', '#EE6677', '#AA3377']

# 方案2 - 高对比度但不刺眼的配色
colors = ['#0077BB', '#EE7733', '#009988', '#CC3311', '#33BBEE', '#EE3377']

NATURE_CONFIG = {
    "width": 1800,
    "height": 900,
    "font_base": 28,
    "font_title": 32,
    "font_axis": 24,
    "font_legend": 24,
    "dpi": 300
}

In [40]:
### a simple logger
logger.remove()
logger.add(sys.stderr, level='DEBUG')

env_seed = 56        #10-h  5-h 0-l 1-h 2-l 3-l 4l 7h 8h 9l 6l

agent_num = 5
agent_policy_net = []
safe_agent_net = []

### create testing environment
injection_bus = np.array([18, 21, 30, 45, 53])-1
pp_net = create_56bus()
env = VoltageCtrl_Env(pp_net, injection_bus)
# state, topology, senario = env.reset_topo(seed=env_seed)      #change topology
state, topology, senario = env.reset(seed=env_seed)             #not change
topology = torch.cuda.FloatTensor(topology).unsqueeze(0)

def moving_average(a, n=3):
    # Padding the array to maintain the length after convolution.
    pad = np.pad(a, (n//2, n-1-n//2), mode='edge')
    ret = np.convolve(pad, np.ones(n), mode='valid') / n
    return ret


### load nn model parameter from saved model 
for i in range(agent_num):
    topology_net = TopologyNet(topology_dim=55, output_dim=1, hidden_dim=Config.topology_hidden_dim)
    policy_net = FlexiblePolicyNet(env=env, topology_net=topology_net, obs_dim=1, action_dim=1, hidden_dim=Config.hidden_dim_56bus).to(device)
    agent_policy_net.append(policy_net)

for i in range(agent_num):
    policy_net = SafePolicyNetwork(env=env, obs_dim=1, action_dim=1, hidden_dim=100).to(device)
    safe_agent_net.append(policy_net)

for i in range(agent_num):
    #value_net_dict = torch.load(f'check_points/value_net/2023-06-19/Step_200_Seed_12_a{i}.pth')
    #policy_net_dict = torch.load(f'check_points/policy_net/2023-08-09/Step_250_Seed_23_a{i}.pth')
    #policy_net_dict = torch.load(f'check_points/policy_net/2023-08-15/Step_900_Seed_33_a{i}.pth')
    #policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2023-09-21/Step_900_Seed_10_a{i}.pth'))
    policy_net_dict = torch.load(os.path.join(Config.data_path,f'check_points/policy_net/2025-02-18/Step_500_Seed_4_a{i}.pth'))

    agent_policy_net[i].load_state_dict(policy_net_dict)

for i in range(agent_num):
    #value_net_dict = torch.load(f'D:/Code/Python/StableRL_VoltageCtrl-main/saved_models/2023-06-19/SafeDDPG_value_Step_200_a{i}.pth')
    policy_net_dict = torch.load(f'D:/Code/Python/StableRL_VoltageCtrl-main/saved_models/stable_ddpg/policy_net_checkpoint_a{i}.pth')

    safe_agent_net[i].load_state_dict(policy_net_dict)

2025-07-28 17:22:59.014 | DEBUG    | Environment:reset:409 - Episode start at high volatage, highest is 1.141656613375231 pu...


In [41]:
episode_reward = 0
episode_control = 0
voltage = []
q = []
cost = []

last_action = np.zeros((agent_num,1))

done_record = True
state, topology, senario = env.reset_topo(seed=env_seed)      #change topology
# state, topology, senario = env.reset(seed=env_seed)             #not change
topology = torch.cuda.FloatTensor(topology).unsqueeze(0)
for t in range(100):
    action = []
    for i in range(agent_num):
        action_agent = agent_policy_net[i](torch.cuda.FloatTensor(state[i].reshape(1,)).unsqueeze(0), topology)
        action_agent = action_agent.detach().cpu().numpy()[0]
        action.append(action_agent)

    if np.min(action) < -1.0 or np.max(action) > 1.0:
        logger.warning('control output saturated! min is {}, max is {}', np.min(action), np.max(action))

    action = last_action - 1.2* np.asarray(action)
    
    last_action = np.copy(action)
    
    try:
        next_state, reward, done = env.step(action)
    except:
        logger.error(sys.exc_info())
        logger.error('power flow not converge at {}', t)
        break

    if done and done_record:
        logger.info('RLC-FT stable at step {}', t)
        logger.info('RLC-FT stable cost is {}', episode_control)
        done_record = False

    voltage.append(state)

    q.append(action)

    state = next_state
    
    episode_reward += reward
    
    cost.append(-reward)
    
    episode_control += LA.norm(action, 2)

    # if done:
    #     break

voltage_RL = np.asarray(voltage)
q_RL =  np.asarray(q)
cost_RL =  np.asarray(cost)
logger.info('control cost of flexible controller is {}',episode_control)
logger.info('objective of flexible controller is {}', np.sum(cost_RL))

### test the base line controller
state, topology, senario = env.reset_topo(seed=env_seed)      #change topology
# state, topology, senario = env.reset(seed=env_seed)             #not change
episode_reward = 0
episode_control = 0
num_agent = 5
voltage = []
q = []
cost = []

last_action = np.zeros((num_agent,1))
done_record = True
for t in range(100):
    state1 = np.asarray(state-env.vmax)
    state2 = np.asarray(env.vmin-state)
    d_v = (np.maximum(state1, 0)-np.maximum(state2, 0)).reshape((num_agent,1))
    
    action = (last_action - 10*d_v)
    
    last_action = np.copy(action)
    
    try:
        next_state, reward, done = env.step(action)
    except:
        logger.error(sys.exc_info())
        logger.error('power flow not converge at {}', t)
        break

    if done and done_record:
        logger.info('Linear stable at step {}', t)
        logger.info('Linear stable cost is {}', episode_control)
        done_record = False

    voltage.append(state)

    q.append(action)

    state = next_state
    
    episode_reward += reward
    
    cost.append(-reward)
    
    episode_control += LA.norm(action, 2)

    # if done:
    #     break

voltage_baseline = np.asarray(voltage)
q_baseline =  np.asarray(q)
cost_baseline =  np.asarray(cost)
logger.info('control cost of linear controller is {}',episode_control)
logger.info('objective of linear controller is {}', np.sum(cost_baseline))

### test the safe policy net
state, topology, senario = env.reset_topo(seed=env_seed)      #change topology
# state, topology, senario = env.reset(seed=env_seed)             #not change
episode_reward = 0
episode_control = 0
num_agent = 5
safe_voltage = []
safe_q = []
safe_cost = []

last_action = np.zeros((num_agent,1))
done_record = True
for t in range(100):
    action = []
    for i in range(num_agent):
        action_agent = safe_agent_net[i].get_action(torch.cuda.FloatTensor([state[i]]).float().reshape(1,1))
        action.append(action_agent)

    if 5*np.min(action) < -1.0 or 5*np.max(action) > 1.0:
        logger.warning('control output saturated! min is {}, max is {}', 5*np.min(action), 5*np.max(action))
    
    action = last_action - 5*np.asarray(action).reshape((num_agent, 1))
    
    last_action = np.copy(action)
    
    try:
        next_state, reward, done = env.step(action)
    except:
        logger.error(sys.exc_info())
        logger.error('power flow not converge at {}', t)
        break

    if done and done_record:
        logger.info('Safe-DDPG stable at step {}', t)
        logger.info('Safe-DDPG stable cost is {}', episode_control)
        done_record = False

    safe_voltage.append(state)

    safe_q.append(action)

    state = next_state
    
    episode_reward += reward
    
    safe_cost.append(-reward)
    
    episode_control += LA.norm(action, 2)

    # if done:
    #     break

safe_voltage = np.asarray(safe_voltage)
safe_q =  np.asarray(safe_q)
safe_cost =  np.asarray(safe_cost)
logger.info('control cost of safe-DDPG is {}',episode_control)
logger.info('objective of linear controller is {}', np.sum(safe_cost))

2025-07-28 17:22:59.287 | WARNING  | __main__:<module>:21 - control output saturated! min is 0.12095801532268524, max is 1.2069556713104248
2025-07-28 17:22:59.341 | WARNING  | __main__:<module>:21 - control output saturated! min is 0.07140974700450897, max is 1.0094014406204224
2025-07-28 17:22:59.622 | INFO     | __main__:<module>:35 - RLC-FT stable at step 6
2025-07-28 17:22:59.624 | INFO     | __main__:<module>:36 - RLC-FT stable cost is 29.222686808792904
2025-07-28 17:23:03.454 | INFO     | __main__:<module>:57 - control cost of flexible controller is 705.0992786569518
2025-07-28 17:23:03.455 | INFO     | __main__:<module>:58 - objective of flexible controller is 1467.455022988397
2025-07-28 17:23:05.074 | INFO     | __main__:<module>:89 - Linear stable at step 98
2025-07-28 17:23:05.076 | INFO     | __main__:<module>:90 - Linear stable cost is 591.874575981545
2025-07-28 17:23:05.091 | INFO     | __main__:<module>:111 - control cost of linear controller is 604.9567113467966
2025

In [83]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots
import numpy as np
from scipy.interpolate import PchipInterpolator, interp1d  # For smooth_curve
import os
# Assuming Config is defined; replace with your path if needed

# ---------------------------
# Figure: Enhanced Nature-Style Voltage Trajectory Comparison
# ---------------------------

# Bus indices and titles (5 buses)
bus_indices = [0, 1, 2, 3, 4]
bus_titles = ['Bus 18', 'Bus 21', 'Bus 30', 'Bus 45', 'Bus 53']

# Create 3x2 subplot layout with improved spacing
fig = make_subplots(
    rows=3, cols=2,
    subplot_titles=[f'(a) {bus_titles[0]}', f'(b) {bus_titles[1]}', 
                    f'(c) {bus_titles[2]}', f'(d) {bus_titles[3]}',
                    f'(e) {bus_titles[4]}', f'(f) Average Voltage'],
    vertical_spacing=0.10,
    horizontal_spacing=0.12,
    shared_xaxes=True,
    shared_yaxes=False  # Independent y for harmony
)
# update font size of all subtitle
for annotation in fig['layout']['annotations']:
    annotation['font'] = dict(size=NATURE_CONFIG['font_base']-4)


# Color and marker scheme for distinction
method_colors = {
    'RLC-FT': '#0072B2',      # Deeper blue for emphasis
    'Linear': '#999999',      # Gray
    'Safe-DDPG': '#D55E00'    # Orange-red
}
method_dashes = {
    'RLC-FT': 'solid',
    'Linear': 'dash',         # Longer dashes for continuity
    'Safe-DDPG': 'dot'        # Shorter dots
}
method_markers = {
    'RLC-FT': 'circle',
    'Linear': 'square',
    'Safe-DDPG': 'triangle-up'
}
line_width = 3.0  # Slightly thicker for visibility
marker_size = 8
marker_opacity = 0.9  # Higher for distinction

method_names = {
    'RLC-FT': 'RLC-FT',
    'Linear': 'Linear',
    'Safe-DDPG': 'Safe-DDPG'
}

# Steps (discrete, limited to 0-30)
steps = np.arange(voltage_RL.shape[0])
display_mask = steps <= 30
display_steps = steps[display_mask]

# Helper to convert hex to rgba
def hex_to_rgba(hex_color, alpha=1.0):
    hex_color = hex_color.lstrip('#')
    r = int(hex_color[0:2], 16)
    g = int(hex_color[2:4], 16)
    b = int(hex_color[4:6], 16)
    return f'rgba({r},{g},{b},{alpha})'

# Smooth curve function for average plot
def smooth_curve(x, y, num_points=200):
    x = np.array(x)
    y = np.array(y)
    mask = x <= 15  # Align with x range [0,15]
    x, y = x[mask], y[mask]
    if len(x) < 2:
        return x, y
    x_new = np.linspace(x.min(), x.max(), num_points)
    try:
        pchip = PchipInterpolator(x, y)
        y_new = pchip(x_new)
    except:
        f = interp1d(x, y, kind='cubic')
        y_new = f(x_new)
    return x_new, y_new

# Precompute voltage data (scaled to kV) and independent y ranges
methods = ['RLC-FT', 'Linear', 'Safe-DDPG']
voltage_data = {
    'RLC-FT': voltage_RL * 12,
    'Linear': voltage_baseline * 12,
    'Safe-DDPG': safe_voltage * 12
}

# Compute averages, mins, maxs for average plot
avg_data = {}
y_ranges = {}  # For independent y ranges
for method in methods:
    data = voltage_data[method][display_mask, :]
    avg = np.mean(data, axis=1)
    min_v = np.min(data, axis=1)
    max_v = np.max(data, axis=1)
    avg_data[method] = {'avg': avg, 'min': min_v, 'max': max_v}

# Compute per-panel y ranges (min/max across methods + padding)
for panel_idx in range(6):
    if panel_idx == 5:  # Average
        all_y = np.concatenate([avg_data[m]['avg'] for m in methods])
        y_min, y_max = np.min(all_y) - 0.2, np.max(all_y) + 0.2
    else:
        bus_idx = bus_indices[panel_idx]
        all_y = np.concatenate([voltage_data[m][display_mask, bus_idx] for m in methods])
        y_min, y_max = np.min(all_y) - 0.2, np.max(all_y) + 0.2
    y_ranges[panel_idx] = [y_min, y_max]

# Add traces to subplots
for panel_idx in range(6):
    row = (panel_idx // 2) + 1
    col = (panel_idx % 2) + 1
    is_average = (panel_idx == 5)
    
    # Add voltage limits (upper solid, lower dashed)
    limit_color = '#666666'  # Subtle gray
    fig.add_trace(
        go.Scatter(x=[0, 15], y=[12.6, 12.6], mode='lines',
                   line=dict(color=limit_color, width=3.0, dash='solid'),
                   showlegend=(panel_idx == 0), name='Upper limit'),
        row=row, col=col
    )
    # fig.add_trace(
    #     go.Scatter(x=[0, 15], y=[12.0, 12.0], mode='lines',
    #                line=dict(color=limit_color, width=1.0, dash='dash'),
    #                showlegend=(panel_idx == 0), name='Lower limit'),
    #     row=row, col=col
    # )
    
    # # Add nominal voltage line
    # fig.add_trace(
    #     go.Scatter(x=[0, 15], y=[12.0, 12.0], mode='lines',
    #                line=dict(color='#BBBBBB', width=0.8, dash='solid'),
    #                showlegend=(panel_idx == 0), name='Nominal voltage'),
    #     row=row, col=col
    # )
    
    if is_average:
        # Average: Smooth curves + subtle min-max shade
        for method_key in methods:
            avg = avg_data[method_key]['avg']
            min_v = avg_data[method_key]['min']
            max_v = avg_data[method_key]['max']
            
            # Smooth all
            x_smooth, avg_smooth = smooth_curve(display_steps, avg)
            _, min_smooth = smooth_curve(display_steps, min_v)
            _, max_smooth = smooth_curve(display_steps, max_v)
            
            # Subtle shade
            fig.add_trace(
                go.Scatter(x=np.concatenate([x_smooth, x_smooth[::-1]]),
                           y=np.concatenate([max_smooth, min_smooth[::-1]]),
                           fill='toself', fillcolor=hex_to_rgba(method_colors[method_key], 0.1),
                           line=dict(color='rgba(0,0,0,0)'),
                           showlegend=False, hoverinfo='skip'),
                row=row, col=col
            )
            
            # Main smooth line
            fig.add_trace(
                go.Scatter(x=x_smooth, y=avg_smooth, mode='lines',
                           line=dict(color=method_colors[method_key], dash=method_dashes[method_key], width=line_width),
                           name=method_names[method_key], showlegend=(panel_idx == 0)),
                row=row, col=col
            )
    else:
        # Individual buses: Straight folded lines + distinct markers
        bus_idx = bus_indices[panel_idx]
        for method_key in methods:
            y_values = voltage_data[method_key][display_mask, bus_idx]
            
            fig.add_trace(
                go.Scatter(x=display_steps, y=y_values, mode='lines+markers',
                           marker=dict(symbol=method_markers[method_key], size=marker_size,
                                       color=method_colors[method_key], opacity=marker_opacity),
                           line=dict(color=method_colors[method_key], dash=method_dashes[method_key], width=line_width),
                           name=method_names[method_key], showlegend=(panel_idx == 0)),
                row=row, col=col
            )
    
    # Axis settings (independent y, subtle grid)
    fig.update_xaxes(range=[0, 15], showgrid=True, gridcolor='#F0F0F0', gridwidth=0.5, zeroline=False,
                     dtick=5,  # Finer ticks
                     title_text="Iteration steps" if row == 3 else None,
                     title_font=dict(size=NATURE_CONFIG['font_axis']), tickfont=dict(size=NATURE_CONFIG['font_axis']-4),
                     row=row, col=col)
    fig.update_yaxes(range=y_ranges[panel_idx], showgrid=True, gridcolor='#F0F0F0', gridwidth=0.5, zeroline=False,
                     dtick=0.3,  # Finer ticks
                     title_text="Voltage (kV)" if col == 1 else None,
                     title_font=dict(size=NATURE_CONFIG['font_axis']), tickfont=dict(size=NATURE_CONFIG['font_axis']-4),
                     row=row, col=col)

# Add minimal annotations (only in first subplot for guidance)
fig.add_annotation(
    x=12, y=12.65, text="Upper limit", showarrow=True, arrowhead=2, arrowsize=1.0, arrowwidth=2,
    font=dict(size=18, color='#666666'), row=2, col=1
)
# fig.add_annotation(
#     x=12, y=11.95, text="Nominal / Lower limit", showarrow=True, arrowhead=1, arrowsize=0.5, arrowwidth=1,
#     font=dict(size=10, color='#666666'), row=1, col=1
# )

# Overall layout (enhanced for readability and Nature style)
fig.update_layout(
    font=dict(family='Arial', size=NATURE_CONFIG['font_base']),
    width=1200,
    height=900,  # Taller for better proportion
    margin=dict(l=60, r=40, t=100, b=60),  # Adjusted for space
    legend=dict(orientation="h", yanchor="bottom", y=1.05, xanchor="center", x=0.5,
                font=dict(size = NATURE_CONFIG['font_legend'])),
    plot_bgcolor='white',
    paper_bgcolor='white',
    showlegend=True
)

# Export high-quality images
fig.write_image(os.path.join(Config.data_path, 'images', '56bus', 'nature_enhanced_trajectory.pdf'), scale=4)
fig.write_image(os.path.join(Config.data_path, 'images', '56bus', 'nature_enhanced_trajectory.png'), scale=2)

fig.show()